In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install pandas openpyxl openai scikit-learn sentence-transformers numpy

In [ ]:
import pandas as pd
import os
import openai
from google.colab import userdata
import json
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import warnings
import numpy as np

# REMOVED: NLTK and ROUGE_SCORE imports as requested
# import nltk
# from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
# from rouge_score import rouge_scorer

# --- Configuration and Setup ---
project_path = '/content/drive/MyDrive/MoE'
excel_data_path = os.path.join(project_path, 'My Data.xlsx') # Path to your data file
output_excel_path = os.path.join(project_path, "Results.xlsx") # Updated output filename

if not os.path.exists(project_path):
    print(f"Error: Project path '{project_path}' does not exist. Please create it or adjust the path.")
    try:
        os.makedirs(project_path)
        print(f"Created project directory: {project_path}")
    except Exception as e:
        print(f"Failed to create project directory: {e}. Please create it manually.")
        exit()

# Initialize SentenceTransformer model for embeddings
try:
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print("SentenceTransformer model loaded successfully.")
except Exception as e:
    print(f"Error loading SentenceTransformer model: {e}. "
          "Automated similarity scores may not work correctly.")
    embedding_model = None

# --- 2. Load Data and Build Vector Store ---
def load_data_and_build_vector_store(excel_path, embedding_model):
    print(f"Attempting to load data from: {excel_path}")
    if not os.path.exists(excel_path):
        print(f"Error: Data file '{excel_path}' not found. Please ensure it's in the correct path.")
        return [], None

    try:
        df = pd.read_excel(excel_path)
        print(f"Loaded {len(df)} rows from '{excel_path}'.")

        if 'non-inclusive' not in df.columns or 'inclusive' not in df.columns:
            print("Error: 'non-inclusive' or 'inclusive' columns not found in the Excel file.")
            return [], None

        contexts = []
        embeddings = []
        for index, row in df.iterrows():
            original_sentence = str(row['non-inclusive']).strip()
            revised_sentence = str(row['inclusive']).strip()
            category = str(row.get('category', 'N/A')).strip() # Ensure category is string and stripped

            if original_sentence and revised_sentence and embedding_model:
                try:
                    embedding = embedding_model.encode(original_sentence, convert_to_tensor=False)
                    embeddings.append(embedding)
                    contexts.append({
                        "original": original_sentence,
                        "revised": revised_sentence,
                        "category": category
                    })
                except Exception as e:
                    print(f"Warning: Could not create embedding for row {index}: {e}")
            elif not original_sentence or not revised_sentence:
                print(f"Warning: Skipping row {index} due to empty 'non-inclusive' or 'inclusive' content.")

        embeddings_array = np.array(embeddings) if embeddings else None
        print(f"Generated embeddings for {len(embeddings)} sentences.")
        return contexts, embeddings_array

    except Exception as e:
        print(f"Error loading or processing Excel data: {e}")
        return [], None

# Load your data and build the vector store when the script starts
all_contexts_data, all_embeddings_array = load_data_and_build_vector_store(excel_data_path, embedding_model)
if not all_contexts_data or all_embeddings_array is None:
    print("WARNING: Vector store could not be initialized from My Data.xlsx. Experts will run without dynamic few-shot examples.")
    # Fallback to dummy contexts if real data loading fails
    def get_top5_contexts(query, **kwargs):
        print("Using DUMMY contexts as My Data.xlsx could not be loaded/processed.")
        return [
            {"original": "The manager's team is very cohesive.", "revised": "The team members work cohesively.", "category": "General"},
            {"original": "We need to hire a strong leader.", "revised": "We need to hire a capable leader.", "category": "Leadership"},
            {"original": "She's a female CEO.", "revised": "She's a CEO.", "category": "Gender"},
            {"original": "The old woman said...", "revised": "The elderly woman said...", "category": "Age"},
            {"original": "The disabled person...", "revised": "The person with a disability...", "category": "Disability"}
        ]
else:
    def get_top5_contexts(query, all_contexts_data=all_contexts_data, all_embeddings_array=all_embeddings_array, embedding_model=embedding_model, k=5):
        if not embedding_model or all_embeddings_array is None or not all_contexts_data:
            print("Cannot retrieve contexts: embedding model or vector store not initialized.")
            return []

        try:
            query_embedding = embedding_model.encode(query, convert_to_tensor=False)
            similarities = cosine_similarity(query_embedding.reshape(1, -1), all_embeddings_array)[0]
            top_k_indices = similarities.argsort()[-k:][::-1]
            top_contexts = [all_contexts_data[i] for i in top_k_indices]
            print(f"Retrieved {len(top_contexts)} relevant contexts for query: '{query[:50]}...'")
            return top_contexts
        except Exception as e:
            print(f"Error during context retrieval: {e}")
            return []

def format_context_examples(contexts):
    """Formats context examples for inclusion in the prompt."""
    if not contexts:
        return ""
    formatted = "### Contextual Examples (for Few-Shot Learning):\n"
    for i, context in enumerate(contexts):
        formatted += f"Example {i+1} (Category: {context.get('category', 'N/A')}):\n"
        formatted += f"  Original: \"{context.get('original', 'N/A')}\"\n"
        formatted += f"  Revised: \"{context.get('revised', 'N/A')}\"\n"
    return formatted + "\n"

def format_contexts_for_excel(contexts):
    """Formats the list of contexts into a readable string for an Excel cell."""
    if not contexts:
        return "No contexts used."
    formatted_string = ""
    for i, context in enumerate(contexts):
        formatted_string += f"Context {i+1} (Category: {context.get('category', 'N/A')}):\n"
        formatted_string += f"  Original: \"{context.get('original', 'N/A')}\"\n"
        formatted_string += f"  Revised: \"{context.get('revised', 'N/A')}\"\n"
        if i < len(contexts) - 1:
            formatted_string += "---\n"
    return formatted_string.strip()



In [ ]:
# --- 3. OpenAI API Configuration and GPT Response Generation ---
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    openai.api_key = OPENAI_API_KEY
    GPT_MODEL_NAME = "gpt-4o-mini"
    print(f"OpenAI API configured with model: {GPT_MODEL_NAME}")

except userdata.SecretNotFoundError:
    print("Error: OPENAI_API_KEY not found in Colab Secrets.")
    print("Please go to 'Secrets' (key icon on left sidebar) and add your API key named 'OPENAI_API_KEY'.")
    exit()
except Exception as e:
    print(f"An error occurred during OpenAI API configuration: {e}")
    exit()

def generate_response_gpt(prompt, max_tokens=512, temperature=0.7, response_format=None):
    """Generates a raw response from the GPT model, with optional JSON response format."""
    try:
        messages = [
            {"role": "user", "content": prompt}
        ]

        if response_format:
            response = openai.chat.completions.create(
                model=GPT_MODEL_NAME,
                messages=messages,
                max_tokens=max_tokens,
                n=1,
                stop=None,
                temperature=temperature,
                response_format=response_format
            )
        else:
            response = openai.chat.completions.create(
                model=GPT_MODEL_NAME,
                messages=messages,
                max_tokens=max_tokens,
                n=1,
                stop=None,
                temperature=temperature,
            )
        generated_text = response.choices[0].message.content.strip()
        return generated_text

    except openai.APIError as e:
        print(f"OpenAI API Error in generate_response_gpt: {e}")
        return f"API Error: {e}"
    except Exception as e:
        print(f"Error generating response with GPT: {e}")
        return f"Error: {e}"


# --- 4. MoE Experts (Encoder) with CoT and Few-Shot Examples ---
def inclusivity_expert_gpt(user_input, contexts):
    context_str = format_context_examples(contexts)
    prompt = (
        f"**Role:** You are an Inclusivity Expert specializing in corporate communications.\n"
        f"**Task:** Analyze the provided original sentence for any language that is biased, discriminatory, exclusive, or reinforcing of harmful stereotypes. Your sole purpose is to make the sentence fully inclusive and respectful for all audiences.\n"
        f"**Instruction:**\n"
        f"1.  **Analyze (Chain of Thought):** First, identify *all* specific non-inclusive phrases or underlying biases in the original sentence. Explain *why* they are non-inclusive. Think step-by-step.\n"
        f"2.  **Revise:** Based on your analysis, rewrite the sentence to be completely inclusive, using neutral, person-first, and universally applicable language. Ensure the core meaning is preserved unless it directly conflicts with inclusivity.\n"
        f"3.  **Prioritize:** If maintaining the original sentence's specific meaning conflicts with inclusivity, prioritize inclusivity.\n"
        f"4.  **Output Format:** Provide your thought process followed by the revised sentence. Use a clear marker for the revised sentence.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{user_input}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=400, temperature=0.3)
    if "Revised Sentence (Inclusive):" in response:
        return response.split("Revised Sentence (Inclusive):")[-1].strip()
    return response

def neutrality_expert_gpt(user_input, contexts):
    context_str = format_context_examples(contexts)
    prompt = (
        f"**Role:** You are a Neutrality Expert specializing in corporate communications.\n"
        f"**Task:** Analyze the provided original sentence for any subjective, emotionally charged, judgmental, presumptive, or biased language. Your sole purpose is to make the sentence entirely objective, factual, and neutral in tone.\n"
        f"**Instruction:**\n"
        f"1.  **Analyze (Chain of Thought):** First, identify *all* specific non-neutral or biased phrases in the original sentence. Explain *why* they are non-neutral. Think step-by-step.\n"
        f"2.  **Revise:** Based on your analysis, rewrite the sentence to be completely neutral, objective, and factual. Remove opinions, assumptions, and emotional language.\n"
        f"3.  **Prioritize:** Focus strictly on neutrality. Ensure information is conveyed without implicit judgment or stereotypes.\n"
        f"4.  **Output Format:** Provide your thought process followed by the revised sentence. Use a clear marker for the revised sentence.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{user_input}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=400, temperature=0.3)
    if "Revised Sentence (Neutral):" in response:
        return response.split("Revised Sentence (Neutral):")[-1].strip()
    return response

def tone_coherence_expert_gpt(user_input, contexts):
    context_str = format_context_examples(contexts)
    prompt = (
        f"**Role:** You are a Tone and Coherence Expert specializing in corporate communications.\n"
        f"**Task:** Analyze the provided original sentence for issues related to professionalism, clarity, conciseness, grammar, spelling, flow, and overall tone. Your sole purpose is to enhance its professional tone and ensure optimal coherence for a corporate audience.\n"
        f"**Instruction:**\n"
        f"1.  **Analyze (Chain of Thought):** First, identify *all* specific issues related to tone, clarity, conciseness, grammar, spelling, or flow in the original sentence. Explain *why* they are issues. Think step-by-step.\n"
        f"2.  **Revise:** Based on your analysis, rewrite the sentence to be highly professional, clear, concise, grammatically correct, and coherent. Ensure it flows naturally and is appropriate for corporate communication.\n"
        f"3.  **Prioritize:** Maintain the original meaning and intent as much as possible, while focusing on professional tone and coherence.\n"
        f"4.  **Output Format:** Provide your thought process followed by the revised sentence. Use a clear marker for the revised sentence.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{user_input}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=400, temperature=0.3)
    if "Revised Sentence (Polished Tone & Coherence):" in response:
        return response.split("Revised Sentence (Polished Tone & Coherence):")[-1].strip()
    return response

# --- 5. MoE Decoder (Refiner) - Modified for Context Analysis ---
def moe_decoder_expert_gpt(original_input, expert_outputs, contexts_used):
    expert_feedback = ""
    for expert_name, revised_text in expert_outputs.items():
        if expert_name and isinstance(revised_text, str) and \
           not revised_text.startswith("API Error") and not revised_text.startswith("Error:") and revised_text:
            expert_feedback += f"- {expert_name}: \"{revised_text}\"\n"

    if not expert_feedback:
        return "No valid revisions could be generated by the experts for refinement.", "No contexts were available or used effectively."

    context_str = format_context_examples(contexts_used)

    prompt = (
        f"**Role:** You are the Mixture of Experts (MoE) Decoder for corporate communication refinement.\n"
        f"**Task:** Synthesize the original sentence and the revisions provided by multiple specialized experts. Your goal is to produce a single, final, refined sentence that is the best possible version, prioritizing **inclusivity and neutrality** above all else, while ensuring **professional tone, coherence, conciseness, and contextual relevance**.\n"
        f"**Instruction:**\n"
        f"1.  **Review Inputs:** Carefully read the ORIGINAL SENTENCE, each expert's REVISED SENTENCE, and the provided CONTEXTUAL EXAMPLES.\n"
        f"2.  **Synthesize and Prioritize (Chain of Thought):** Think step-by-step. First, analyze how each expert's revision addresses the original sentence's issues. Prioritize addressing inclusivity and neutrality problems. Then, consider tone, clarity, and conciseness. Explain your reasoning for combining or choosing certain aspects of the revisions.\n"
        f"3.  **Context Influence Analysis (Chain of Thought):** After forming your refined sentence, specifically analyze *how* the provided '{len(contexts_used)} contextual examples' (if any) influenced your decision-making and the final refinement. Did they provide direct phrasing, stylistic guidance, or conceptual clarity? Explain their specific helpfulness.\n"
        f"4.  **Refine & Polish:** Perform any final polishing to make the sentence fluent and impactful for corporate use, maintaining the core factual meaning.\n"
        f"5.  **Output Format:** Provide your complete thought process, followed by a clear marker for the final refined sentence, and then a clear marker for the context influence analysis. Do not include any other commentary or extraneous text.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{original_input}\n\n"
        f"**Expert Revisions:**\n{expert_feedback}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=600, temperature=0.5)

    final_refined_sentence = "Error: Could not extract refined sentence."
    context_influence_analysis = "Error: Could not extract context influence analysis."

    if "Final Refined Sentence:" in response:
        parts_after_sentence_marker = response.split("Final Refined Sentence:", 1)
        if len(parts_after_sentence_marker) > 1:
            content_after_sentence = parts_after_sentence_marker[1].strip()
            if "Context Influence Analysis:" in content_after_sentence:
                sentence_and_analysis_parts = content_after_sentence.split("Context Influence Analysis:", 1)
                final_refined_sentence = sentence_and_analysis_parts[0].strip()
                context_influence_analysis = sentence_and_analysis_parts[1].strip()
            else:
                final_refined_sentence = content_after_sentence
                context_influence_analysis = "No explicit 'Context Influence Analysis:' marker found in response. Analysis may be integrated into thought process."
        else:
            final_refined_sentence = "Error: 'Final Refined Sentence:' marker found, but no content after it."
    else:
        final_refined_sentence = "Error: 'Final Refined Sentence:' marker not found. Full response:\n" + response
        context_influence_analysis = "No explicit 'Context Influence Analysis:' marker found in response. Full response:\n" + response

    return final_refined_sentence, context_influence_analysis


# --- 6. Evaluation Expert (for detailed scores) - Updated to 0/1/2 Scale ---
def evaluation_expert_gpt_detailed(original_sentence, refined_sentence):
    if not isinstance(refined_sentence, str) or not refined_sentence or refined_sentence.startswith("Error"):
        return {
            "Inclusivity_Score": "N/A (Refinement Error)",
            "Quality_Score": "N/A (Refinement Error)",
            "Relevance_Score": "N/A (Refinement Error)",
            "Coherence_Score": "N/A (Refinement Error)"
        }

    inclusivity_criteria = (
        "0: Contains significant biased, discriminatory, or exclusive language. Makes inclusivity worse or shows no attempt.\n"
        "1: Shows some effort towards inclusivity but still contains minor biases, awkward phrasing, or misses key opportunities for improvement. Acceptable, but not optimal.\n"
        "2: **Excellent**. Fully inclusive, uses neutral, person-first, and universally applicable language. Completely free from bias, discrimination, or stereotypes. Model example of inclusive language."
    )
    quality_criteria = (
        "0: Poor quality. Contains severe grammatical errors, misspellings, awkward phrasing, or is difficult to understand. Does not meet professional standards.\n"
        "1: Acceptable quality. Contains minor grammatical issues, slight awkwardness, or could be more concise. Meets basic professional standards but has room for improvement.\n"
        "2: **Excellent**. High professional quality. Flawless grammar, spelling, punctuation. Clear, concise, and effectively communicates the message. Polished and professional."
    )
    relevance_criteria = (
        "0: Irrelevant. Significantly changes the original meaning, introduces irrelevant information, or is completely off-topic. Core intent is lost.\n"
        "1: Partially relevant. Preserves some of the original meaning but may omit important details, add slightly irrelevant information, or introduce minor inaccuracies. Core intent is partially maintained.\n"
        "2: **Excellent**. Fully relevant. Preserves the exact original meaning and intent. No information is lost, and no irrelevant details are introduced. Perfectly maintains context and factual accuracy."
    )
    coherence_criteria = (
        "0: Incoherent. Disjointed, illogical flow, difficult to follow, or lacks clear connections between ideas.\n"
        "1: Partially coherent. Generally understandable but may have awkward transitions, unnatural phrasing, or require re-reading to grasp meaning. Flow could be smoother.\n"
        "2: **Excellent**. Highly coherent. Logical flow, smooth transitions, easy to read and understand. Ideas are clearly connected, and the sentence feels natural and well-structured."
    )

    prompt = (
        f"**Role:** You are a highly precise and objective Linguistic Evaluation Expert.\n"
        f"**Task:** Assess the provided REVISED SENTENCE against the ORIGINAL SENTENCE based on four specific aspects: Inclusivity, Quality, Relevance, and Coherence. Your evaluation must strictly adhere to the provided 0/1/2 scoring criteria for each aspect.\n"
        f"**Instruction:**\n"
        f"1.  **Read Carefully:** Analyze both sentences in the context of each requested aspect.\n"
        f"2.  **Apply Criteria STRICTLY (0/1/2 Scale):** For each aspect, assign a score from 0 to 2. Match the REVISED SENTENCE's quality *exactly* to the description for that score. **DO NOT give a score of 2 unless ALL conditions for a '2' are perfectly met.** Assign a 0 if the revised sentence makes the issue significantly worse or completely fails on that aspect.\n"
        f"3.  **Justify Concisely:** For *each* score, provide a brief, concise, and specific justification that explains *why* the given score was assigned. Reference specific words or phrases from the sentences if applicable to support your score.\n"
        f"4.  **Output Format:** Your response MUST be a single, valid JSON object with the following structure, containing all four scores and justifications. Do not include any other text:\n"
        f"    ```json\n"
        f"    {{\n"
        f"        \"inclusivity\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_inclusivity_score>\"\n"
        f"        }},\n"
        f"        \"quality\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_quality_score>\"\n"
        f"        }},\n"
        f"        \"relevance\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_relevance_score>\"\n"
        "        }},\n"
        f"        \"coherence\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_coherence_score>\"\n"
        f"        }}\n"
        f"    }}\n"
        f"    ```\n"
        f"5.  **Strictness:** Maintain absolute objectivity. Do not infer intent or make assumptions. Evaluate *only* based on the textual evidence provided and the defined criteria. Be critical and precise.\n\n"
        f"--- Data Sample to Evaluate ---\n"
        f"Original Sentence: \"{original_sentence}\"\n"
        f"Revised Sentence: \"{refined_sentence}\"\n\n"
        f"### Evaluation Request (Provide JSON Output Here):"
    )

    try:
        response_json_str = generate_response_gpt(prompt, max_tokens=400, temperature=0.2, response_format={"type": "json_object"})
        llm_output = json.loads(response_json_str)

        scores = {}
        for aspect in ["inclusivity", "quality", "relevance", "coherence"]:
            if aspect in llm_output and isinstance(llm_output[aspect], dict):
                score_value = llm_output[aspect].get("score")
                if isinstance(score_value, (int, float)) and 0 <= score_value <= 2:
                    scores[f"{aspect.replace(' ', '_')}_Score"] = score_value
                else:
                    scores[f"{aspect.replace(' ', '_')}_Score"] = f"N/A (Invalid Score '{score_value}')"
            else:
                scores[f"{aspect.replace(' ', '_')}_Score"] = "N/A (Missing Aspect in JSON)"

        expected_score_keys = ["Inclusivity_Score", "Quality_Score", "Relevance_Score", "Coherence_Score"]
        for expected_key in expected_score_keys:
            if expected_key not in scores:
                scores[expected_key] = "N/A (Missing from JSON after parsing)"

        return scores
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error in evaluation_expert_gpt_detailed: {e}")
        print(f"Raw LLM response that failed to parse: \n---\n{response_json_str}\n---")
        return {
            "Inclusivity_Score": "N/A (JSON Parse Error)",
            "Quality_Score": "N/A (JSON Parse Error)",
            "Relevance_Score": "N/A (JSON Parse Error)",
            "Coherence_Score": "N/A (JSON Parse Error)"
        }
    except Exception as e:
        print(f"General Error in evaluation_expert_gpt_detailed: {e}")
        return {
            "Inclusivity_Score": "N/A (General Error)",
            "Quality_Score": "N/A (General Error)",
            "Relevance_Score": "N/A (General Error)",
            "Coherence_Score": "N/A (General Error)"
        }


# --- 7. Automated Evaluation Metrics (Non-BLEU/ROUGE) ---
# Removed: calculate_bleu_rouge_scores function

def calculate_automated_scores(original_sentence, expert_outputs):
    scores = {}

    texts_to_evaluate = {
        "Inclusivity Expert (GPT)": expert_outputs.get("Inclusivity Expert (GPT)"),
        "Neutrality Expert (GPT)": expert_outputs.get("Neutrality Expert (GPT)"),
        "Tone & Coherence Expert (GPT)": expert_outputs.get("Tone & Coherence Expert (GPT)"),
        "Final Refined Output (GPT)": expert_outputs.get("Final Refined Output (GPT)")
    }

    for expert_name, output_text in texts_to_evaluate.items():
        if isinstance(output_text, str) and output_text and \
           not output_text.startswith("API Error") and not output_text.startswith("Error:"):
            # Length Change Ratio
            original_len = len(original_sentence.split())
            output_len = len(output_text.split())
            scores[f"{expert_name}_Length_Change_Ratio"] = (output_len / original_len) if original_len > 0 else 0

            # Similarity to Original
            if embedding_model is not None:
                try:
                    original_embedding = embedding_model.encode(original_sentence, convert_to_tensor=False)
                    output_embedding = embedding_model.encode(output_text, convert_to_tensor=False)
                    similarity = cosine_similarity(original_embedding.reshape(1, -1), output_embedding.reshape(1, -1))[0][0]
                    scores[f"{expert_name}_Similarity_to_Original"] = similarity.item()
                except Exception as e:
                    scores[f"{expert_name}_Similarity_to_Original"] = "N/A (Embedding Error)"
            else:
                scores[f"{expert_name}_Similarity_to_Original"] = "N/A (Model Not Loaded)"

            # REMOVED: BLEU and ROUGE Scores
            # bleu_rouge = calculate_bleu_rouge_scores(original_sentence, output_text)
            # for metric, value in bleu_rouge.items():
            #     scores[f"{expert_name}_{metric}"] = value
        else:
            scores[f"{expert_name}_Length_Change_Ratio"] = "N/A (Error or No Output)"
            scores[f"{expert_name}_Similarity_to_Original"] = "N/A (Error or No Output)"
            # Removed N/A for BLEU/ROUGE
            # scores[f"{expert_name}_BLEU_Score"] = "N/A (Error or No Output)"
            # scores[f"{expert_name}_ROUGE_1_F1"] = "N/A (Error or No Output)"
            # scores[f"{expert_name}_ROUGE_2_F1"] = "N/A (Error or No Output)"
            # scores[f"{expert_name}_ROUGE_L_F1"] = "N/A (Error or No Output)"

    return scores


# --- 8. Main Pipeline Execution Function ---
def run_moe_pipeline_gpt(user_input):
    print(f"--- Running Experts for: \"{user_input}\" ---")

    top5_contexts = get_top5_contexts(user_input)

    inclusive_output = inclusivity_expert_gpt(user_input, top5_contexts)
    print(f"  Inclusivity Expert Output: {inclusive_output}")
    neutral_output = neutrality_expert_gpt(user_input, top5_contexts)
    print(f"  Neutrality Expert Output: {neutral_output}")
    tone_output = tone_coherence_expert_gpt(user_input, top5_contexts)
    print(f"  Tone & Coherence Expert Output: {tone_output}")

    intermediate_results = {
        "Original": user_input,
        "Inclusivity Expert (GPT)": inclusive_output,
        "Neutrality Expert (GPT)": neutral_output,
        "Tone & Coherence Expert (GPT)": tone_output,
        "Top5_Contexts_Used": top5_contexts
    }

    refiner_input_outputs = {
        "Inclusivity Expert": inclusive_output,
        "Neutrality Expert": neutral_output,
        "Tone & Coherence Expert": tone_output
    }
    print("\n--- Running MoE Decoder (Refiner) ---")
    final_refined_output, context_influence_analysis = moe_decoder_expert_gpt(user_input, refiner_input_outputs, top5_contexts)
    print(f"  Final Refined Output (GPT): {final_refined_output}")
    print(f"  Context Influence Analysis: {context_influence_analysis}")
    intermediate_results["Final Refined Output (GPT)"] = final_refined_output
    intermediate_results["Context_Influence_Analysis"] = context_influence_analysis

    print("\n--- Running Detailed Evaluation Expert (LLM-based) ---")
    detailed_evaluation_scores = evaluation_expert_gpt_detailed(user_input, final_refined_output)
    for score_name, score_value in detailed_evaluation_scores.items():
        print(f"  LLM_Score_{score_name}: {score_value}")
        intermediate_results[f"LLM_Score_{score_name}"] = score_value

    return intermediate_results

# --- 9. Test Cases and Main Execution Loop (with Excel Export) ---
#test_inputs = [
  #  "Despite her humble background, the female engineer managed to solve the complex algorithm, proving that even people from disadvantaged places can contribute.",
   # "The manager, a true visionary, believes that empowering his team members, regardless of their age or physical condition, leads to better productivity. He always says, 'Every individual brings unique skills to the table, and we must cater to the needs of each and every one, even the disabled, to ensure no one feels left out during our intensive all-day meetings.'",
   #"It's a really stupid idea to bring someone from a different background into this team; they'll just make things more chaotic and we'll have to babysit them. We need a strong hand to lead the charge, not some indecisive, overly emotional leader.",
   #"The old guard in the finance department struggles with modern software, so we're hiring young, tech-savvy grads to handle the real work and avoid any slowdowns during our high-pressure quarterly reports.",
   #"Our groundbreaking new policy ensures that all men are guaranteed equal opportunities in leadership, as they are naturally more decisive and less emotional when making tough calls, particularly in stressful business negotiations.",
   #"We need someone with strong leadership qualities, like a real man, to run this aggressive new sales campaign, especially considering the current market downturn. A woman would simply not be able to handle the pressure and maintain a competitive edge.",
 # "The homeless person sleeping outside the building is a nuisance; we should find a way to make them move, as their presence negatively impacts our professional image and deters potential clients.",
  #"Everyone should come to the charity gala; it's only a $500 ticket, so it's very affordable, and it's for a great cause. We expect all the wealthy patrons to attend and contribute generously.",
  #"Our new hire from India is great at math, as expected. It's always good to have someone from that region on the data analytics team; they just seem to have a natural talent for numbers.",
 # "The company's social initiative aims to help underprivileged children in inner-city areas, as they are often lacking in basic education and opportunities, unlike those from more privileged suburban communities.",
  #"Our new female engineering lead, despite her petite stature, demonstrated a surprisingly strong grasp of the technical challenges, really holding her own in the meeting with the male-dominated senior team. It's refreshing to see women step up in these demanding technical fields.",
  #"While we appreciate the enthusiasm of our fresh-faced new graduates, it's clear their initial ideas for the digital transformation project, though innovative, lack the practical experience our seasoned veterans bring to the table. We need solutions that are grounded in reality, not just youthful idealism.",
  #"Our recent community outreach program successfully equipped children from the inner-city neighborhoods with basic coding skills, a crucial step given their limited access to technology. We're proud to bridge this gap, offering them a rare chance to catch up with their more privileged peers.",
  #"The prostitute body was found in an alley",
  #"Although women are generally more emotional and less logical than men, they tend to perform adequately in nurturing professions like nursing or teaching, even though their lack of assertiveness might hinder them in leadership roles.",
  #"While women can be good team players, they often struggle with the kind of tough decision-making required in upper management, where a more assertive and less emotional approach is necessary to drive results.",
  #"Older employees, although loyal and reliable, usually find it difficult to adapt to fast-paced tech environments, which are better suited for younger professionals who are naturally more innovative and tech-savvy.",
  #"Applicants from certain communities tend to be too informal or overly expressive during interviews, which can come across as unprofessional compared to the clear and composed demeanor typically expected in corporate environments.",
  #"Fathers who take parental leave may enjoy bonding with their child, but since mothers are naturally better at nurturing, it’s generally more appropriate for them to take on the primary caregiving responsibilities.",
  #"Students from underprivileged backgrounds often lack the discipline and preparedness that elite schools require, so admission policies should prioritize those from families who can better support academic excellence.",
  #"For someone of her generation, she’s surprisingly adaptable to new technologies and eager to engage with younger colleagues, which makes her a unique asset on agile teams.",
  #"Even with his condition, he’s managed to consistently meet deadlines and maintain a positive attitude, which is truly commendable given the circumstances.",
  #"Her leadership style is collaborative and emotionally intelligent, which brings a refreshing contrast to the usual results-driven approach typical of executive roles."]
#test_inputs = [
 #   "Our new female engineering lead, despite her petite stature, demonstrated a surprisingly strong grasp of the technical challenges, really holding her own in the meeting with the male-dominated senior team. It's refreshing to see women step up in these demanding technical fields.",
  #  "While we appreciate the enthusiasm of our fresh-faced new graduates, it's clear their initial ideas for the digital transformation project, though innovative, lack the practical experience our seasoned veterans bring to the table. We need solutions that are grounded in reality, not just youthful idealism.",
   # "Our recent community outreach program successfully equipped children from the inner-city neighborhoods with basic coding skills, a crucial step given their limited access to technology. We're proud to bridge this gap, offering them a rare chance to catch up with their more privileged peers.",
    #"The prostitute body was found in an alley",
    #"Although women are generally more emotional and less logical than men, they tend to perform adequately in nurturing professions like nursing or teaching, even though their lack of assertiveness might hinder them in leadership roles, which require a firm hand and strategic thinking—traits more commonly associated with male executives who are naturally more driven and analytical.",
    #"While women can be good team players, they often struggle with the kind of tough decision-making required in upper management, where a more assertive and less emotional approach is necessary to drive results."
#]
test_inputs = [
 #   "For someone from a traditionally conservative background, Maya's openness about her same-sex partner is a bold step, and while it may not sit well with all clients, it certainly demonstrates her confidence.",
  #  "Despite being a young Black man from an inner-city school, Darius has shown remarkable discipline and professionalism — qualities that are essential when interacting with our high-profile clientele.",
   # "While her limited mobility means she works remotely, Ana still manages to contribute meaningfully to the cultural diversity initiatives, which is commendable considering her condition and lack of in-person interaction.",
    #"Given their non-binary identity and unconventional dress style, Alex has integrated surprisingly well into our corporate culture, showing that adaptability can come from even the most unexpected candidates.",
    #"Though English isn't her first language, and she comes from a rural migrant community, Priya expresses herself clearly in most situations, which has helped her keep up with the pace of the team.",
    #"As an older gay man in tech, Roberto has managed to stay current with trends, and his presence on the team brings a much-needed alternative perspective, even if his leadership style differs from the younger majority.",
    #"Being a first-generation college student and the child of refugees, Sam has exceeded expectations — especially in maintaining a professional demeanor that aligns well with our company’s brand image.",
    #"Although born with a cognitive disability, Jerome has surprised many by handling structured tasks well, particularly when he sticks to routines and avoids high-pressure social situations.",
    #"Considering her Muslim faith and traditional upbringing, it's encouraging that Aisha is willing to take part in mixed-gender leadership discussions, even if she prefers to remain somewhat reserved during debates.",
    #"Despite coming from a marginalized indigenous background and lacking early access to formal education, Lani has managed to present herself as knowledgeable in cross-cultural contexts, which is valuable in international meetings.",

    # --- NEW SUPER-DUPER HARD MODE TEST CASES ---

    # 11. Bias Combination: Ageism + Physical Appearance + Gender + Professional Competence
    #"For a woman her age, her vibrant outfit and active participation in the morning jog club prove she's still got the energy to lead her department, a surprise for many who expect managers to slow down.",

    # 12. Bias Combination: Disability (Neurodivergence) + Professional Role + Implicit Scrutiny + Systemic Blame
   # "The data analyst on the spectrum delivered a precise report, a result that’s consistent for someone who struggles with social cues, demonstrating how well he can focus when left undisturbed by team discussions.",

    # 13. Bias Combination: Race/Ethnicity + Socioeconomic Status + Education/Intellect + Soft Bigotry of Low Expectations
  #  "Despite having grown up in a historically under-resourced community and attending only local public schools, Chen has developed surprisingly strong analytical skills, a testament to individual effort over systemic opportunities.",

    # 14. Bias Combination: LGBTQ+ Identity (Transgender) + Caregiving Role + Implicit Prejudice + Social Acceptance
  #  "As a trans man, Mark’s dedication to being a stay-at-home parent is truly admirable, especially as it challenges traditional gender roles and may not always be understood by wider community circles.",

    # 15. Bias Combination: Religion + Geographic/Cultural Origin + Perceived Threat/Othering + Professional Role
 #   "The security consultant, who is originally from a Middle Eastern country and openly practices Islam, maintained a neutral demeanor during the intense political discussion, assuaging initial concerns about his impartiality."
#]
# test_inputs = ["Black women are always so loud and aggressive.",
    "Asian girls are submissive and good at following orders.",
    "Latina women are fiery but not really professional.",
    "You don’t act like a typical Indian man — you’re actually respectful.",
    "Single moms on welfare just want handouts.",
    "Working-class women should just focus on raising kids, not careers.",
    "She’s disabled and still managed to look good today — that’s impressive.",
    "Dating a disabled woman would be too much work.",
    "A Muslim woman shouldn’t be in politics — she’s too biased.",
    "You’re too pretty to be a hijabi — why do you hide your beauty?",
    "I’m surprised a Black autistic kid could be that smart.",
    "He’s blind and African — that’s a double disadvantage.",
    "Gay Black men are always so flamboyant.",
    "You’re trans and Latina? That’s a lot to deal with.",
    "Don’t you feel oppressed wearing that turban?",
    "People from your country are always too extreme with their beliefs.",
    "Middle-aged Arab men always look suspicious at the airport."
    "She’s surprisingly articulate for someone who grew up in that neighborhood.",
    "I didn’t expect a Black woman to lead such a technical project.",
    "He’s doing well, especially considering he’s on the spectrum and from a working-class background.",
    "You’re a gay engineer? That’s an unusual combination.",
    "She handles the finances well — not something you see often with single moms.",
    "It's great that a Latina woman like her is interested in classical literature.",
    "I would have never guessed he was disabled — he seems so professional.",
    "They’ve done an impressive job, especially being trans and from a low-income family.",
    "He’s more polite than I expected from someone with his background.",
    "A woman of her age and race managing a whole department — that’s rare.",
    "She's actually good at coding — didn’t expect that from a fashion major.",
    "He’s blind, but you can barely tell in meetings — he really holds his own.",
    "She doesn’t seem like the activist type, even though she wears a hijab.",
    "He’s openly gay but still quite masculine in how he presents.",
    "Even with her disability, she managed to complete law school — that’s really admirable.",
    "It's refreshing to see someone from a rural background speak so well in public.",
    "For a Muslim woman, she’s remarkably confident and outspoken in leadership roles.",
    "I didn’t expect someone from that country to understand Western workplace culture so well.",
    "She’s a trans woman in tech and she actually knows her stuff.",
    "Despite being on a scholarship and from a tough area, he blends in just fine here."]
all_results_for_excel = []

if os.path.exists(output_excel_path):
    print(f"Existing Excel file found at: {output_excel_path}.")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            df_existing = pd.read_excel(output_excel_path)
        starting_id = df_existing['Test_Case_ID'].max() + 1 if not df_existing.empty else 1
        all_results_for_excel.extend(df_existing.to_dict('records'))
        print(f"Loaded {len(df_existing)} existing records. New test cases will start from ID {starting_id}.")
    except Exception as e:
        print(f"Error loading existing Excel file: {e}. Starting with a new file from ID 1.")
        starting_id = 1
else:
    print(f"No existing Excel file found at: {output_excel_path}. Creating a new file from ID 1.")
    starting_id = 1


print("\n===== Running MoE with GPT Experts, Decoder, and Detailed LLM Evaluation =====")
for i, test_input in enumerate(test_inputs):
    current_test_id = starting_id + i
    print(f"\n===== Test Case {current_test_id} =====")
    print(f"Original: {test_input}")

    results_for_case = run_moe_pipeline_gpt(test_input)

    row_data = {
        "Test_Case_ID": current_test_id,
        "Original_Sentence": test_input,
        "Inclusivity_Expert_Output": results_for_case.get("Inclusivity Expert (GPT)", ""),
        "Neutrality_Expert_Output": results_for_case.get("Neutrality Expert (GPT)", ""),
        "Tone_Coherence_Expert_Output": results_for_case.get("Tone & Coherence Expert (GPT)", ""),
        "Final_Refined_Output_Decoder": results_for_case.get("Final Refined Output (GPT)", ""),
        "Top5_Contexts_Used": format_contexts_for_excel(results_for_case.get("Top5_Contexts_Used", [])),
        "Context_Influence_Analysis": results_for_case.get("Context_Influence_Analysis", "")
    }

    # Add LLM-based scores
    llm_eval_scores = {k: v for k, v in results_for_case.items() if k.startswith("LLM_Score_")}
    for score_name, score_value in llm_eval_scores.items():
        row_data[score_name] = score_value

    # Calculate and add automated (non-LLM) evaluation scores
    automated_scores = calculate_automated_scores(test_input, results_for_case)
    for metric, value in automated_scores.items():
        row_data[f"Automated_Score_{metric}"] = value

    all_results_for_excel.append(row_data)

    print("\n--- Summary for Test Case ---")
    for key, value in row_data.items():
        if key not in ["Test_Case_ID", "Original_Sentence"]:
            print(f"{key}: {value}")
    print("-" * 30)

df_final = pd.DataFrame(all_results_for_excel)
df_final.to_excel(output_excel_path, index=False)

print(f"\n===== MoE Pipeline Execution Complete =====")
print(f"Results exported to: {output_excel_path}")
print("You can find the Excel file in your Google Drive at the specified path.")

OpenAI API configured with model: gpt-4o-mini
Existing Excel file found at: /content/drive/MyDrive/MoE/Results.xlsx.
Loaded 15 existing records. New test cases will start from ID 16.

===== Running MoE with GPT Experts, Decoder, and Detailed LLM Evaluation =====

===== Test Case 16 =====
Original: Asian girls are submissive and good at following orders.
--- Running Experts for: "Asian girls are submissive and good at following orders." ---
Retrieved 5 relevant contexts for query: 'Asian girls are submissive and good at following o...'
  Inclusivity Expert Output: 1. **Analysis:**
   - The phrase "Asian girls" is problematic because it generalizes a specific racial and gender group, which can reinforce stereotypes. It implies that all individuals within this group share certain traits, which is not true and disregards individual differences.
   - The term "submissive" is a harmful stereotype that suggests passivity and lack of agency, which can perpetuate negative views about women, par

In [ ]:
!pip install -q google-generativeai


In [ ]:
# --- 3. OpenAI API Configuration and GPT Response Generation ---
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    openai.api_key = OPENAI_API_KEY
    GPT_MODEL_NAME = "gpt-4o-mini"
    print(f"OpenAI API configured with model: {GPT_MODEL_NAME}")

except userdata.SecretNotFoundError:
    print("Error: OPENAI_API_KEY not found in Colab Secrets.")
    print("Please go to 'Secrets' (key icon on left sidebar) and add your API key named 'OPENAI_API_KEY'.")
    exit()
except Exception as e:
    print(f"An error occurred during OpenAI API configuration: {e}")
    exit()

def generate_response_gpt(prompt, max_tokens=512, temperature=0.7, response_format=None):
    """Generates a raw response from the GPT model, with optional JSON response format."""
    try:
        messages = [
            {"role": "user", "content": prompt}
        ]

        if response_format:
            response = openai.chat.completions.create(
                model=GPT_MODEL_NAME,
                messages=messages,
                max_tokens=max_tokens,
                n=1,
                stop=None,
                temperature=temperature,
                response_format=response_format
            )
        else:
            response = openai.chat.completions.create(
                model=GPT_MODEL_NAME,
                messages=messages,
                max_tokens=max_tokens,
                n=1,
                stop=None,
                temperature=temperature,
            )
        generated_text = response.choices[0].message.content.strip()
        return generated_text

    except openai.APIError as e:
        print(f"OpenAI API Error in generate_response_gpt: {e}")
        return f"API Error: {e}"
    except Exception as e:
        print(f"Error generating response with GPT: {e}")
        return f"Error: {e}"


# --- 4. MoE Experts (Encoder) with CoT and Few-Shot Examples ---
def inclusivity_expert_gpt(user_input, contexts):
    context_str = format_context_examples(contexts)
    prompt = (
        f"**Role:** You are an Inclusivity Expert specializing in corporate communications.\n"
        f"**Task:** Analyze the provided original sentence for any language that is biased, discriminatory, exclusive, or reinforcing of harmful stereotypes. Your sole purpose is to make the sentence fully inclusive and respectful for all audiences.\n"
        f"**Instruction:**\n"
        f"1.  **Analyze (Chain of Thought):** First, identify *all* specific non-inclusive phrases or underlying biases in the original sentence. Explain *why* they are non-inclusive. Think step-by-step.\n"
        f"2.  **Revise:** Based on your analysis, rewrite the sentence to be completely inclusive, using neutral, person-first, and universally applicable language. Ensure the core meaning is preserved unless it directly conflicts with inclusivity.\n"
        f"3.  **Prioritize:** If maintaining the original sentence's specific meaning conflicts with inclusivity, prioritize inclusivity.\n"
        f"4.  **Output Format:** Provide your thought process followed by the revised sentence. Use a clear marker for the revised sentence.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{user_input}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=400, temperature=0.3)
    if "Revised Sentence (Inclusive):" in response:
        return response.split("Revised Sentence (Inclusive):")[-1].strip()
    return response

def neutrality_expert_gpt(user_input, contexts):
    context_str = format_context_examples(contexts)
    prompt = (
        f"**Role:** You are a Neutrality Expert specializing in corporate communications.\n"
        f"**Task:** Analyze the provided original sentence for any subjective, emotionally charged, judgmental, presumptive, or biased language. Your sole purpose is to make the sentence entirely objective, factual, and neutral in tone.\n"
        f"**Instruction:**\n"
        f"1.  **Analyze (Chain of Thought):** First, identify *all* specific non-neutral or biased phrases in the original sentence. Explain *why* they are non-neutral. Think step-by-step.\n"
        f"2.  **Revise:** Based on your analysis, rewrite the sentence to be completely neutral, objective, and factual. Remove opinions, assumptions, and emotional language.\n"
        f"3.  **Prioritize:** Focus strictly on neutrality. Ensure information is conveyed without implicit judgment or stereotypes.\n"
        f"4.  **Output Format:** Provide your thought process followed by the revised sentence. Use a clear marker for the revised sentence.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{user_input}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=400, temperature=0.3)
    if "Revised Sentence (Neutral):" in response:
        return response.split("Revised Sentence (Neutral):")[-1].strip()
    return response

def tone_coherence_expert_gpt(user_input, contexts):
    context_str = format_context_examples(contexts)
    prompt = (
        f"**Role:** You are a Tone and Coherence Expert specializing in corporate communications.\n"
        f"**Task:** Analyze the provided original sentence for issues related to professionalism, clarity, conciseness, grammar, spelling, flow, and overall tone. Your sole purpose is to enhance its professional tone and ensure optimal coherence for a corporate audience.\n"
        f"**Instruction:**\n"
        f"1.  **Analyze (Chain of Thought):** First, identify *all* specific issues related to tone, clarity, conciseness, grammar, spelling, or flow in the original sentence. Explain *why* they are issues. Think step-by-step.\n"
        f"2.  **Revise:** Based on your analysis, rewrite the sentence to be highly professional, clear, concise, grammatically correct, and coherent. Ensure it flows naturally and is appropriate for corporate communication.\n"
        f"3.  **Prioritize:** Maintain the original meaning and intent as much as possible, while focusing on professional tone and coherence.\n"
        f"4.  **Output Format:** Provide your thought process followed by the revised sentence. Use a clear marker for the revised sentence.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{user_input}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=400, temperature=0.3)
    if "Revised Sentence (Polished Tone & Coherence):" in response:
        return response.split("Revised Sentence (Polished Tone & Coherence):")[-1].strip()
    return response

# --- 5. MoE Decoder (Refiner) - Modified for Context Analysis ---
def moe_decoder_expert_gpt(original_input, expert_outputs, contexts_used):
    expert_feedback = ""
    for expert_name, revised_text in expert_outputs.items():
        if expert_name and isinstance(revised_text, str) and \
           not revised_text.startswith("API Error") and not revised_text.startswith("Error:") and revised_text:
            expert_feedback += f"- {expert_name}: \"{revised_text}\"\n"

    if not expert_feedback:
        return "No valid revisions could be generated by the experts for refinement.", "No contexts were available or used effectively."

    context_str = format_context_examples(contexts_used)

    prompt = (
        f"**Role:** You are the Mixture of Experts (MoE) Decoder for corporate communication refinement.\n"
        f"**Task:** Synthesize the original sentence and the revisions provided by multiple specialized experts. Your goal is to produce a single, final, refined sentence that is the best possible version, prioritizing **inclusivity and neutrality** above all else, while ensuring **professional tone, coherence, conciseness, and contextual relevance**.\n"
        f"**Instruction:**\n"
        f"1.  **Review Inputs:** Carefully read the ORIGINAL SENTENCE, each expert's REVISED SENTENCE, and the provided CONTEXTUAL EXAMPLES.\n"
        f"2.  **Synthesize and Prioritize (Chain of Thought):** Think step-by-step. First, analyze how each expert's revision addresses the original sentence's issues. Prioritize addressing inclusivity and neutrality problems. Then, consider tone, clarity, and conciseness. Explain your reasoning for combining or choosing certain aspects of the revisions.\n"
        f"3.  **Context Influence Analysis (Chain of Thought):** After forming your refined sentence, specifically analyze *how* the provided '{len(contexts_used)} contextual examples' (if any) influenced your decision-making and the final refinement. Did they provide direct phrasing, stylistic guidance, or conceptual clarity? Explain their specific helpfulness.\n"
        f"4.  **Refine & Polish:** Perform any final polishing to make the sentence fluent and impactful for corporate use, maintaining the core factual meaning.\n"
        f"5.  **Output Format:** Provide your complete thought process, followed by a clear marker for the final refined sentence, and then a clear marker for the context influence analysis. Do not include any other commentary or extraneous text.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{original_input}\n\n"
        f"**Expert Revisions:**\n{expert_feedback}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=600, temperature=0.5)

    final_refined_sentence = "Error: Could not extract refined sentence."
    context_influence_analysis = "Error: Could not extract context influence analysis."

    if "Final Refined Sentence:" in response:
        parts_after_sentence_marker = response.split("Final Refined Sentence:", 1)
        if len(parts_after_sentence_marker) > 1:
            content_after_sentence = parts_after_sentence_marker[1].strip()
            if "Context Influence Analysis:" in content_after_sentence:
                sentence_and_analysis_parts = content_after_sentence.split("Context Influence Analysis:", 1)
                final_refined_sentence = sentence_and_analysis_parts[0].strip()
                context_influence_analysis = sentence_and_analysis_parts[1].strip()
            else:
                final_refined_sentence = content_after_sentence
                context_influence_analysis = "No explicit 'Context Influence Analysis:' marker found in response. Analysis may be integrated into thought process."
        else:
            final_refined_sentence = "Error: 'Final Refined Sentence:' marker found, but no content after it."
    else:
        final_refined_sentence = "Error: 'Final Refined Sentence:' marker not found. Full response:\n" + response
        context_influence_analysis = "No explicit 'Context Influence Analysis:' marker found in response. Full response:\n" + response

    return final_refined_sentence, context_influence_analysis


# --- 6. Evaluation Expert (for detailed scores) - Updated to 0/1/2 Scale ---
def evaluation_expert_gpt_detailed(original_sentence, refined_sentence):
    if not isinstance(refined_sentence, str) or not refined_sentence or refined_sentence.startswith("Error"):
        return {
            "Inclusivity_Score": "N/A (Refinement Error)",
            "Quality_Score": "N/A (Refinement Error)",
            "Relevance_Score": "N/A (Refinement Error)",
            "Coherence_Score": "N/A (Refinement Error)"
        }

    inclusivity_criteria = (
        "0: Contains significant biased, discriminatory, or exclusive language. Makes inclusivity worse or shows no attempt.\n"
        "1: Shows some effort towards inclusivity but still contains minor biases, awkward phrasing, or misses key opportunities for improvement. Acceptable, but not optimal.\n"
        "2: **Excellent**. Fully inclusive, uses neutral, person-first, and universally applicable language. Completely free from bias, discrimination, or stereotypes. Model example of inclusive language."
    )
    quality_criteria = (
        "0: Poor quality. Contains severe grammatical errors, misspellings, awkward phrasing, or is difficult to understand. Does not meet professional standards.\n"
        "1: Acceptable quality. Contains minor grammatical issues, slight awkwardness, or could be more concise. Meets basic professional standards but has room for improvement.\n"
        "2: **Excellent**. High professional quality. Flawless grammar, spelling, punctuation. Clear, concise, and effectively communicates the message. Polished and professional."
    )
    relevance_criteria = (
        "0: Irrelevant. Significantly changes the original meaning, introduces irrelevant information, or is completely off-topic. Core intent is lost.\n"
        "1: Partially relevant. Preserves some of the original meaning but may omit important details, add slightly irrelevant information, or introduce minor inaccuracies. Core intent is partially maintained.\n"
        "2: **Excellent**. Fully relevant. Preserves the exact original meaning and intent. No information is lost, and no irrelevant details are introduced. Perfectly maintains context and factual accuracy."
    )
    coherence_criteria = (
        "0: Incoherent. Disjointed, illogical flow, difficult to follow, or lacks clear connections between ideas.\n"
        "1: Partially coherent. Generally understandable but may have awkward transitions, unnatural phrasing, or require re-reading to grasp meaning. Flow could be smoother.\n"
        "2: **Excellent**. Highly coherent. Logical flow, smooth transitions, easy to read and understand. Ideas are clearly connected, and the sentence feels natural and well-structured."
    )

    prompt = (
        f"**Role:** You are a highly precise and objective Linguistic Evaluation Expert.\n"
        f"**Task:** Assess the provided REVISED SENTENCE against the ORIGINAL SENTENCE based on four specific aspects: Inclusivity, Quality, Relevance, and Coherence. Your evaluation must strictly adhere to the provided 0/1/2 scoring criteria for each aspect.\n"
        f"**Instruction:**\n"
        f"1.  **Read Carefully:** Analyze both sentences in the context of each requested aspect.\n"
        f"2.  **Apply Criteria STRICTLY (0/1/2 Scale):** For each aspect, assign a score from 0 to 2. Match the REVISED SENTENCE's quality *exactly* to the description for that score. **DO NOT give a score of 2 unless ALL conditions for a '2' are perfectly met.** Assign a 0 if the revised sentence makes the issue significantly worse or completely fails on that aspect.\n"
        f"3.  **Justify Concisely:** For *each* score, provide a brief, concise, and specific justification that explains *why* the given score was assigned. Reference specific words or phrases from the sentences if applicable to support your score.\n"
        f"4.  **Output Format:** Your response MUST be a single, valid JSON object with the following structure, containing all four scores and justifications. Do not include any other text:\n"
        f"    ```json\n"
        f"    {{\n"
        f"        \"inclusivity\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_inclusivity_score>\"\n"
        f"        }},\n"
        f"        \"quality\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_quality_score>\"\n"
        f"        }},\n"
        f"        \"relevance\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_relevance_score>\"\n"
        "        }},\n"
        f"        \"coherence\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_coherence_score>\"\n"
        f"        }}\n"
        f"    }}\n"
        f"    ```\n"
        f"5.  **Strictness:** Maintain absolute objectivity. Do not infer intent or make assumptions. Evaluate *only* based on the textual evidence provided and the defined criteria. Be critical and precise.\n\n"
        f"--- Data Sample to Evaluate ---\n"
        f"Original Sentence: \"{original_sentence}\"\n"
        f"Revised Sentence: \"{refined_sentence}\"\n\n"
        f"### Evaluation Request (Provide JSON Output Here):"
    )

    try:
        response_json_str = generate_response_gpt(prompt, max_tokens=400, temperature=0.2, response_format={"type": "json_object"})
        llm_output = json.loads(response_json_str)

        scores = {}
        for aspect in ["inclusivity", "quality", "relevance", "coherence"]:
            if aspect in llm_output and isinstance(llm_output[aspect], dict):
                score_value = llm_output[aspect].get("score")
                if isinstance(score_value, (int, float)) and 0 <= score_value <= 2:
                    scores[f"{aspect.replace(' ', '_')}_Score"] = score_value
                else:
                    scores[f"{aspect.replace(' ', '_')}_Score"] = f"N/A (Invalid Score '{score_value}')"
            else:
                scores[f"{aspect.replace(' ', '_')}_Score"] = "N/A (Missing Aspect in JSON)"

        expected_score_keys = ["Inclusivity_Score", "Quality_Score", "Relevance_Score", "Coherence_Score"]
        for expected_key in expected_score_keys:
            if expected_key not in scores:
                scores[expected_key] = "N/A (Missing from JSON after parsing)"

        return scores
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error in evaluation_expert_gpt_detailed: {e}")
        print(f"Raw LLM response that failed to parse: \n---\n{response_json_str}\n---")
        return {
            "Inclusivity_Score": "N/A (JSON Parse Error)",
            "Quality_Score": "N/A (JSON Parse Error)",
            "Relevance_Score": "N/A (JSON Parse Error)",
            "Coherence_Score": "N/A (JSON Parse Error)"
        }
    except Exception as e:
        print(f"General Error in evaluation_expert_gpt_detailed: {e}")
        return {
            "Inclusivity_Score": "N/A (General Error)",
            "Quality_Score": "N/A (General Error)",
            "Relevance_Score": "N/A (General Error)",
            "Coherence_Score": "N/A (General Error)"
        }


# --- 7. Automated Evaluation Metrics (Non-BLEU/ROUGE) ---
# Removed: calculate_bleu_rouge_scores function

def calculate_automated_scores(original_sentence, expert_outputs):
    scores = {}

    texts_to_evaluate = {
        "Inclusivity Expert (GPT)": expert_outputs.get("Inclusivity Expert (GPT)"),
        "Neutrality Expert (GPT)": expert_outputs.get("Neutrality Expert (GPT)"),
        "Tone & Coherence Expert (GPT)": expert_outputs.get("Tone & Coherence Expert (GPT)"),
        "Final Refined Output (GPT)": expert_outputs.get("Final Refined Output (GPT)")
    }

    for expert_name, output_text in texts_to_evaluate.items():
        if isinstance(output_text, str) and output_text and \
           not output_text.startswith("API Error") and not output_text.startswith("Error:"):
            # Length Change Ratio
            original_len = len(original_sentence.split())
            output_len = len(output_text.split())
            scores[f"{expert_name}_Length_Change_Ratio"] = (output_len / original_len) if original_len > 0 else 0

            # Similarity to Original
            if embedding_model is not None:
                try:
                    original_embedding = embedding_model.encode(original_sentence, convert_to_tensor=False)
                    output_embedding = embedding_model.encode(output_text, convert_to_tensor=False)
                    similarity = cosine_similarity(original_embedding.reshape(1, -1), output_embedding.reshape(1, -1))[0][0]
                    scores[f"{expert_name}_Similarity_to_Original"] = similarity.item()
                except Exception as e:
                    scores[f"{expert_name}_Similarity_to_Original"] = "N/A (Embedding Error)"
            else:
                scores[f"{expert_name}_Similarity_to_Original"] = "N/A (Model Not Loaded)"

            # REMOVED: BLEU and ROUGE Scores
            # bleu_rouge = calculate_bleu_rouge_scores(original_sentence, output_text)
            # for metric, value in bleu_rouge.items():
            #     scores[f"{expert_name}_{metric}"] = value
        else:
            scores[f"{expert_name}_Length_Change_Ratio"] = "N/A (Error or No Output)"
            scores[f"{expert_name}_Similarity_to_Original"] = "N/A (Error or No Output)"
            # Removed N/A for BLEU/ROUGE
            # scores[f"{expert_name}_BLEU_Score"] = "N/A (Error or No Output)"
            # scores[f"{expert_name}_ROUGE_1_F1"] = "N/A (Error or No Output)"
            # scores[f"{expert_name}_ROUGE_2_F1"] = "N/A (Error or No Output)"
            # scores[f"{expert_name}_ROUGE_L_F1"] = "N/A (Error or No Output)"

    return scores


# --- 8. Main Pipeline Execution Function ---
def run_moe_pipeline_gpt(user_input):
    print(f"--- Running Experts for: \"{user_input}\" ---")

    top5_contexts = get_top5_contexts(user_input)

    inclusive_output = inclusivity_expert_gpt(user_input, top5_contexts)
    print(f"  Inclusivity Expert Output: {inclusive_output}")
    neutral_output = neutrality_expert_gpt(user_input, top5_contexts)
    print(f"  Neutrality Expert Output: {neutral_output}")
    tone_output = tone_coherence_expert_gpt(user_input, top5_contexts)
    print(f"  Tone & Coherence Expert Output: {tone_output}")

    intermediate_results = {
        "Original": user_input,
        "Inclusivity Expert (GPT)": inclusive_output,
        "Neutrality Expert (GPT)": neutral_output,
        "Tone & Coherence Expert (GPT)": tone_output,
        "Top5_Contexts_Used": top5_contexts
    }

    refiner_input_outputs = {
        "Inclusivity Expert": inclusive_output,
        "Neutrality Expert": neutral_output,
        "Tone & Coherence Expert": tone_output
    }
    print("\n--- Running MoE Decoder (Refiner) ---")
    final_refined_output, context_influence_analysis = moe_decoder_expert_gpt(user_input, refiner_input_outputs, top5_contexts)
    print(f"  Final Refined Output (GPT): {final_refined_output}")
    print(f"  Context Influence Analysis: {context_influence_analysis}")
    intermediate_results["Final Refined Output (GPT)"] = final_refined_output
    intermediate_results["Context_Influence_Analysis"] = context_influence_analysis

    print("\n--- Running Detailed Evaluation Expert (LLM-based) ---")
    detailed_evaluation_scores = evaluation_expert_gpt_detailed(user_input, final_refined_output)
    for score_name, score_value in detailed_evaluation_scores.items():
        print(f"  LLM_Score_{score_name}: {score_value}")
        intermediate_results[f"LLM_Score_{score_name}"] = score_value

    return intermediate_results

# --- 9. Test Cases and Main Execution Loop (with Excel Export) ---
#test_inputs = [
     "Asian girls are submissive and good at following orders.",
    "Latina women are fiery but not really professional.",
    "You don’t act like a typical Indian man — you’re actually respectful.",
    "Single moms on welfare just want handouts.",
    "Working-class women should just focus on raising kids, not careers.",
    "She’s disabled and still managed to look good today — that’s impressive.",
    "Dating a disabled woman would be too much work.",
    "A Muslim woman shouldn’t be in politics — she’s too biased.",
    "You’re too pretty to be a hijabi — why do you hide your beauty?",
    "I’m surprised a Black autistic kid could be that smart.",
    "He’s blind and African — that’s a double disadvantage.",
    "Gay Black men are always so flamboyant.",
    "You’re trans and Latina? That’s a lot to deal with.",
    "Don’t you feel oppressed wearing that turban?",
    "People from your country are always too extreme with their beliefs.",
    "Middle-aged Arab men always look suspicious at the airport."
    "She’s surprisingly articulate for someone who grew up in that neighborhood.",
    "I didn’t expect a Black woman to lead such a technical project.",
    "He’s doing well, especially considering he’s on the spectrum and from a working-class background.",
    "You’re a gay engineer? That’s an unusual combination.",
    "She handles the finances well — not something you see often with single moms.",
    "It's great that a Latina woman like her is interested in classical literature.",
    "I would have never guessed he was disabled — he seems so professional.",
    "They’ve done an impressive job, especially being trans and from a low-income family.",
    "He’s more polite than I expected from someone with his background.",
    "A woman of her age and race managing a whole department — that’s rare.",
    "She's actually good at coding — didn’t expect that from a fashion major.",
    "He’s blind, but you can barely tell in meetings — he really holds his own.",
    "She doesn’t seem like the activist type, even though she wears a hijab.",
    "He’s openly gay but still quite masculine in how he presents.",
    "Even with her disability, she managed to complete law school — that’s really admirable.",
    "It's refreshing to see someone from a rural background speak so well in public.",
    "For a Muslim woman, she’s remarkably confident and outspoken in leadership roles.",
    "I didn’t expect someone from that country to understand Western workplace culture so well.",
    "She’s a trans woman in tech and she actually knows her stuff.",
    "Despite being on a scholarship and from a tough area, he blends in just fine here."]
all_results_for_excel = []

if os.path.exists(output_excel_path):
    print(f"Existing Excel file found at: {output_excel_path}.")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            df_existing = pd.read_excel(output_excel_path)
        starting_id = df_existing['Test_Case_ID'].max() + 1 if not df_existing.empty else 1
        all_results_for_excel.extend(df_existing.to_dict('records'))
        print(f"Loaded {len(df_existing)} existing records. New test cases will start from ID {starting_id}.")
    except Exception as e:
        print(f"Error loading existing Excel file: {e}. Starting with a new file from ID 1.")
        starting_id = 1
else:
    print(f"No existing Excel file found at: {output_excel_path}. Creating a new file from ID 1.")
    starting_id = 1


print("\n===== Running MoE with GPT Experts, Decoder, and Detailed LLM Evaluation =====")
for i, test_input in enumerate(test_inputs):
    current_test_id = starting_id + i
    print(f"\n===== Test Case {current_test_id} =====")
    print(f"Original: {test_input}")

    results_for_case = run_moe_pipeline_gpt(test_input)

    row_data = {
        "Test_Case_ID": current_test_id,
        "Original_Sentence": test_input,
        "Inclusivity_Expert_Output": results_for_case.get("Inclusivity Expert (GPT)", ""),
        "Neutrality_Expert_Output": results_for_case.get("Neutrality Expert (GPT)", ""),
        "Tone_Coherence_Expert_Output": results_for_case.get("Tone & Coherence Expert (GPT)", ""),
        "Final_Refined_Output_Decoder": results_for_case.get("Final Refined Output (GPT)", ""),
        "Top5_Contexts_Used": format_contexts_for_excel(results_for_case.get("Top5_Contexts_Used", [])),
        "Context_Influence_Analysis": results_for_case.get("Context_Influence_Analysis", "")
    }

    # Add LLM-based scores
    llm_eval_scores = {k: v for k, v in results_for_case.items() if k.startswith("LLM_Score_")}
    for score_name, score_value in llm_eval_scores.items():
        row_data[score_name] = score_value

    # Calculate and add automated (non-LLM) evaluation scores
    automated_scores = calculate_automated_scores(test_input, results_for_case)
    for metric, value in automated_scores.items():
        row_data[f"Automated_Score_{metric}"] = value

    all_results_for_excel.append(row_data)

    print("\n--- Summary for Test Case ---")
    for key, value in row_data.items():
        if key not in ["Test_Case_ID", "Original_Sentence"]:
            print(f"{key}: {value}")
    print("-" * 30)

df_final = pd.DataFrame(all_results_for_excel)
df_final.to_excel(output_excel_path, index=False)

print(f"\n===== MoE Pipeline Execution Complete =====")
print(f"Results exported to: {output_excel_path}")
print("You can find the Excel file in your Google Drive at the specified path.")

In [ ]:
import pandas as pd
import re
import textwrap

# --- Configuration ---
FILE_TO_ANALYZE = "/content/drive/My Drive/MoE/test.xlsx" # Your data file. Corrected to .xlsx.
OUTPUT_FILE_NAME = "/content/drive/My Drive/MoE/analysis_output.csv" # Name for the output CSV file

# --- Optional: Semantic Similarity Model (Requires 'sentence-transformers' library) ---
# Uncomment and install if you want to run semantic similarity calculations locally.
# pip install sentence-transformers
try:
    from sentence_transformers import SentenceTransformer, util
    # It's good practice to ensure torch is imported if using sentence_transformers
    import torch
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    SEMANTIC_SIMILARITY_ENABLED = True
    print("Semantic similarity model loaded successfully.")
except ImportError:
    print("Warning: 'sentence_transformers' not found. Semantic similarity analysis will be skipped.")
    print("To enable, install with: pip install sentence-transformers")
    embedding_model = None
    SEMANTIC_SIMILARITY_ENABLED = False
except Exception as e:
    print(f"Error loading semantic similarity model: {e}")
    embedding_model = None
    SEMANTIC_SIMILARITY_ENABLED = False


# --- Utility Functions ---

def parse_context_string(context_string):
    """
    Parses the multi-line context string from a column (e.g., 'Top5_Contexts_Used')
    into a dictionary of original and revised sentences for each context.
    """
    results = {}
    if pd.isna(context_string):
        return results

    # Split by "---" to get individual context blocks
    context_blocks = context_string.strip().split('---')

    for i, block in enumerate(context_blocks):
        if not block.strip():
            continue # Skip empty blocks

        # Use regex to find "Original: " and "Revised: " lines
        # Using re.DOTALL to allow '.' to match newlines for multi-line sentences
        original_match = re.search(r'Original: "(.*?)"', block, re.DOTALL)
        revised_match = re.search(r'Revised: "(.*?)"', block, re.DOTALL)

        context_num = i + 1 # Context numbers are 1-indexed

        results[f'Context{context_num}_Original'] = original_match.group(1).strip() if original_match else None
        results[f'Context{context_num}_Revised'] = revised_match.group(1).strip() if revised_match else None

    return results

def calculate_similarity_scores(row, output_col, context_type, model):
    """
    Calculates semantic similarity between the final output and given contexts.
    'context_type' can be 'Original' or 'Revised'.
    """
    scores = {}
    if not model:
        return scores # Return empty if model not loaded

    final_output_text = row[output_col]
    if pd.isna(final_output_text) or not final_output_text.strip():
        return scores # Cannot calculate if final output is empty

    try:
        final_output_embedding = model.encode(final_output_text, convert_to_tensor=True)

        for i in range(1, 6): # Assuming up to 5 contexts
            context_col = f'Context{i}_{context_type}'
            context_text = row.get(context_col)

            if pd.notna(context_text) and context_text.strip():
                context_embedding = model.encode(context_text, convert_to_tensor=True)
                similarity = util.pytorch_cos_sim(context_embedding, final_output_embedding).item()
                scores[f'{context_col}_Similarity_to_FinalOutput'] = similarity
            else:
                scores[f'{context_col}_Similarity_to_FinalOutput'] = None # No context to compare
    except Exception as e:
        # Catch potential errors during encoding or similarity calculation
        print(f"Error calculating similarity for row {row.name}, context type {context_type}: {e}")
    return scores

# --- Main Execution ---
if __name__ == "__main__":
    try:
        # --- IMPORTANT CHANGE HERE: Use pd.read_excel() for .xlsx files ---
        # Remember to install openpyxl if you haven't: pip install openpyxl
        df = pd.read_excel(FILE_TO_ANALYZE)


        # Ensure essential columns exist
        required_cols = ['Original_Sentence', 'Final_Refined_Output_Decoder', 'Top5_Contexts_Used']
        if not all(col in df.columns for col in required_cols):
            missing_cols = [col for col in required_cols if col not in df.columns]
            print(f"Error: The file '{FILE_TO_ANALYZE}' is missing required columns: {missing_cols}")
            print("Please ensure your Excel file has 'Original_Sentence', 'Final_Refined_Output_Decoder', and 'Top5_Contexts_Used' columns.")
            # Optionally, print available columns to help user debug
            print(f"Available columns in the file: {df.columns.tolist()}")
        else:
            print(f"Successfully loaded '{FILE_TO_ANALYZE}' and found required columns.")

            # Step 1: Parse the 'Top5_Contexts_Used' column
            print("\nStep 1: Parsing 'Top5_Contexts_Used' column...")
            parsed_contexts_df = df['Top5_Contexts_Used'].apply(parse_context_string).apply(pd.Series)
            df = pd.concat([df, parsed_contexts_df], axis=1)
            print("Parsing complete. New context columns added.")

            # Step 2: Calculate Semantic Similarity Scores (if enabled)
            if SEMANTIC_SIMILARITY_ENABLED and embedding_model:
                print("\nStep 2: Calculating semantic similarity scores...")
                # Calculate similarity for 'Original' contexts
                original_sim_scores = df.apply(lambda row: calculate_similarity_scores(row, 'Final_Refined_Output_Decoder', 'Original', embedding_model), axis=1)
                df = pd.concat([df, original_sim_scores.apply(pd.Series)], axis=1)

                # Calculate similarity for 'Revised' contexts
                revised_sim_scores = df.apply(lambda row: calculate_similarity_scores(row, 'Final_Refined_Output_Decoder', 'Revised', embedding_model), axis=1)
                df = pd.concat([df, revised_sim_scores.apply(pd.Series)], axis=1)
                print("Semantic similarity calculations complete.")
            else:
                print("\nStep 2: Semantic similarity calculations skipped (model not available or error during loading).")


            # --- Display Analysis Results for the first few rows (for console output) ---
            print("\n" + "="*100)
            print("Analysis of Top 5 Contexts with Final Output - First 5 Rows (Console Preview):")
            print("="*100)

            # Select and print relevant columns for console display
            display_columns_for_console = ['Original_Sentence', 'Final_Refined_Output_Decoder', 'Top5_Contexts_Used']
            for i in range(1, 6):
                display_columns_for_console.append(f'Context{i}_Original')
                display_columns_for_console.append(f'Context{i}_Revised')
                if SEMANTIC_SIMILARITY_ENABLED:
                    display_columns_for_console.append(f'Context{i}_Original_Similarity_to_FinalOutput')
                    display_columns_for_console.append(f'Context{i}_Revised_Similarity_to_FinalOutput')

            # Filter for columns that actually exist
            existing_display_columns_for_console = [col for col in display_columns_for_console if col in df.columns]

            # Print a summary table with relevant columns
            print(df[existing_display_columns_for_console].head().to_string(max_colwidth=80, line_width=120))


            # --- Save the complete DataFrame to a CSV file ---
            print(f"\nSaving the complete analyzed data to '{OUTPUT_FILE_NAME}'...")
            df.to_csv(OUTPUT_FILE_NAME, index=False)
            print(f"Data successfully saved to '{OUTPUT_FILE_NAME}'.")


            # --- Conceptual/Qualitative Analysis Guide ---
            print("\n" + "="*100)
            print("Guide for Conceptual/Qualitative Analysis:")
            print("="*100)
            print("For each row, consider the following points to understand context contribution:")
            print("1.  **Direct Learning:** Compare 'Original_Sentence' with 'Final_Refined_Output_Decoder'. Does the change mirror any 'ContextX_Original' -> 'ContextX_Revised' example directly? This is strong evidence of direct learning.")
            print("2.  **Principle Application:** Even if not a direct copy, does the change in 'Final_Refined_Output_Decoder' adhere to the *general principle* shown in a context? (e.g., gender-neutral language, avoiding stereotypes, promoting inclusivity).")
            print("3.  **Semantic Closeness (if scores available):** Look at the `_Similarity_to_FinalOutput` scores. Higher scores for a specific context (especially its 'Revised' form) indicate that the final output's meaning is strongly aligned with that context. This suggests a strong influence.")
            print("4.  **Bias Identification & Correction:** Does the `Original_Sentence` contain a bias that is explicitly addressed by one of the `ContextX_Original` examples? If the `Final_Refined_Output_Decoder` corrects that bias, it suggests the context helped in identification and transformation.")
            print("\n**For automated, detailed qualitative explanations, you can load the saved CSV file and use an external powerful LLM (e.g., Gemini, GPT-4) with appropriate prompts for specific rows.**")
            print("="*100)


    except FileNotFoundError:
        print(f"Error: The file '{FILE_TO_ANALYZE}' was not found. Please ensure it's in the correct directory, or the path is accurate.")
    except Exception as e:
        print(f"An unexpected error occurred during execution: {e}")

Semantic similarity model loaded successfully.
Successfully loaded '/content/drive/My Drive/MoE/test.xlsx' and found required columns.

Step 1: Parsing 'Top5_Contexts_Used' column...
Parsing complete. New context columns added.

Step 2: Calculating semantic similarity scores...
Semantic similarity calculations complete.

Analysis of Top 5 Contexts with Final Output - First 5 Rows (Console Preview):
                                                                 Original_Sentence  \
0  Despite her humble background, the female engineer managed to solve the comp...   
1  It's a really stupid idea to bring someone from a different background into ...   
2  The old guard in the finance department struggles with modern software, so w...   
3  Our groundbreaking new policy ensures that all men are guaranteed equal oppo...   
4  We need someone with strong leadership qualities, like a real man, to run th...   

                                                      Final_Refined_Output_Decode